In [5]:
# ============================================================
# CHALLENGE 2 — Fake News Detection
# Model: roberta-base | Metric: Overall Accuracy
# ============================================================

!pip install transformers datasets scikit-learn -q

In [7]:
import random, numpy as np, pandas as pd, torch
random.seed(42); np.random.seed(42); torch.manual_seed(42)

In [8]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [9]:
# ── 1. Load Data ──────────────────────────────────────────
train = pd.read_csv("fakenews_with_labels.csv",
                    encoding='latin-1', on_bad_lines='skip')
test  = pd.read_csv("FakeNews_no_labels.csv",
                    encoding='latin-1', on_bad_lines='skip')

In [10]:
# Fix BOM character in title column
train.columns = train.columns.str.strip().str.replace('ÿ', '', regex=False)
test.columns  = test.columns.str.strip().str.replace('ÿ', '', regex=False)

print("Train columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())
print("Train shape:", train.shape)
print("Label distribution:\n", train['label'].value_counts())

Train columns: ['title', 'text', 'subject', 'date', 'label']
Test columns: ['title', 'text', 'subject', 'date', 'label']
Train shape: (7863, 5)
Label distribution:
 label
True     4372
False    3491
Name: count, dtype: int64


In [11]:
# ── 2. Clean Data ─────────────────────────────────────────
# Drop nulls
train = train.dropna(subset=['label']).reset_index(drop=True)
train['title'] = train['title'].fillna('')
train['text']  = train['text'].fillna('')
test['title']  = test['title'].fillna('')
test['text']   = test['text'].fillna('')

# Combine title + text + subject (max signal)
train['combined'] = (train['title'] + ' [SEP] ' +
                     train['text']  + ' [SEP] ' +
                     train['subject'].fillna(''))
test['combined']  = (test['title']  + ' [SEP] ' +
                     test['text']   + ' [SEP] ' +
                     test['subject'].fillna(''))


In [12]:
# Truncate to 512 words max (BERT limit)
train['combined'] = train['combined'].apply(lambda x: ' '.join(x.split()[:400]))
test['combined']  = test['combined'].apply(lambda x: ' '.join(x.split()[:400]))

TEXT_COL  = 'combined'
LABEL_COL = 'label'

print("\nSample combined text:", train['combined'].iloc[0][:200])


Sample combined text: VIDEO: HARLEM BAR Kicks Customers Out For Wearing Trump Hats: â¬SWe donâ¬"t play that sh*t hereâ¬ [SEP] A large group of very diverse young adults who call themselves The Modern Patriots videotaped t


In [13]:
# ── 3. Encode Labels ──────────────────────────────────────
# Labels are "True" / "False" strings
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train['label_enc'] = le.fit_transform(train[LABEL_COL])
print("\nLabel mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
# False=0, True=1



Label mapping: {np.False_: np.int64(0), np.True_: np.int64(1)}


In [14]:
# ── 4. Baseline — TF-IDF + Logistic Regression ───────────
print("\n--- Running Baseline ---")
vec = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X = vec.fit_transform(train[TEXT_COL])
y = train['label_enc']

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
scores = cross_val_score(lr, X, y, cv=5, scoring='accuracy')
print(f"Baseline CV Accuracy: {scores.mean():.4f}")


--- Running Baseline ---
Baseline CV Accuracy: 0.9893


In [15]:
# Save baseline submission
lr.fit(X, y)
baseline_preds_enc = lr.predict(vec.transform(test[TEXT_COL]))
baseline_preds = le.inverse_transform(baseline_preds_enc)

test_baseline = pd.read_csv("FakeNews_no_labels.csv",
                             encoding='latin-1', on_bad_lines='skip')
test_baseline.columns = test_baseline.columns.str.strip().str.replace('ÿ', '', regex=False)
test_baseline['label'] = baseline_preds
test_baseline.to_csv("FakeNews_no_labels_baseline.csv", index=False)
print("✅ Baseline submission saved")

✅ Baseline submission saved


In [16]:
# ── 5. Main Model — RoBERTa ───────────────────────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset

MODEL = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tokenizer(batch[TEXT_COL], truncation=True,
                     padding="max_length", max_length=256)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [17]:
# Build HF Dataset
train_ds = Dataset.from_pandas(
    train[[TEXT_COL, 'label_enc']].rename(columns={'label_enc': 'label'})
)
train_ds = train_ds.map(tokenize, batched=True)
split = train_ds.train_test_split(test_size=0.1, seed=42)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

args = TrainingArguments(
    output_dir="./c2_results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    seed=42,
    logging_steps=100,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
)
trainer.train()

Map:   0%|          | 0/7863 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.021148,0.038151
2,0.012785,0.010844
3,0.000157,0.011748


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1329, training_loss=0.019763679534443523, metrics={'train_runtime': 324.4811, 'train_samples_per_second': 65.421, 'train_steps_per_second': 4.096, 'total_flos': 2792660741591040.0, 'train_loss': 0.019763679534443523, 'epoch': 3.0})

In [26]:
#  6. Evaluation
val_preds = trainer.predict(split['test'])
val_labels_pred = np.argmax(val_preds.predictions, axis=1)
val_labels_true = split['test']['label']

print("\n" + "="*50)
print("VALIDATION RESULTS")
print("="*50)
print(classification_report(
    val_labels_true, val_labels_pred,
    target_names=[str(c) for c in le.classes_]
))
print(f"Accuracy: {accuracy_score(val_labels_true, val_labels_pred):.4f}")
print("="*50)


VALIDATION RESULTS
              precision    recall  f1-score   support

       False       1.00      1.00      1.00       332
        True       1.00      1.00      1.00       455

    accuracy                           1.00       787
   macro avg       1.00      1.00      1.00       787
weighted avg       1.00      1.00      1.00       787

Accuracy: 0.9987


In [22]:
# ── 7. Predict & Save Final Submission ───────────────────
test_ds = Dataset.from_pandas(test[[TEXT_COL]])
test_ds = test_ds.map(tokenize, batched=True)

final_preds = trainer.predict(test_ds)
final_enc   = np.argmax(final_preds.predictions, axis=1)
final_labels = le.inverse_transform(final_enc)  # "True" / "False"

Map:   0%|          | 0/1541 [00:00<?, ? examples/s]

In [23]:
# Fill label column — keep original structure
test_final = pd.read_csv("FakeNews_no_labels.csv",
                          encoding='latin-1', on_bad_lines='skip')
test_final.columns = test_final.columns.str.strip().str.replace('ÿ', '', regex=False)
test_final['label'] = final_labels


In [25]:
# Verify
print("\nPredicted label values:", test_final['label'].value_counts().to_dict())
print("Expected: True / False")
print("Null labels:", test_final['label'].isnull().sum())

test_final.to_csv("FakeNews_no_labels.csv", index=False)


Predicted label values: {True: 842, False: 699}
Expected: True / False
Null labels: 0


In [27]:
from google.colab import files
files.download("FakeNews_no_labels.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>